# Классификация DDoS-атак с помощью MLP

Ниже — пошаговый пайплайн: от загрузки датасета до оценки качества на тесте.

**Что делаем:**
- читаем и очищаем данные;
- кодируем целевую метку;
- балансируем train-часть;
- обучаем MLP с early stopping;
- считаем метрики на тестовой выборке.

In [2]:
# Базовые библиотеки для обучения MLP-классификатора
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Инструменты предобработки и оценки качества
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Основные параметры эксперимента
FILE_PATH = 'sample_data/DDoSdata.csv'
TARGET_COL = 'attack'          # 'attack', 'category' или 'subcategory'
RESAMPLING_METHOD = 'smote'    # none | undersample | oversample | smote
BATCH_SIZE = 64
NUM_EPOCHS = 10
PATIENCE = 10
RANDOM_STATE = 42

## Подготовка данных

В этом блоке загружаем CSV, выбираем числовые признаки, кодируем target и делим выборку на train/val/test.

In [3]:
print("Загрузка данных...")
df = pd.read_csv(FILE_PATH, low_memory=False)

# Приводим порты к числам (некорректные значения уйдут в NaN)
for col in ['sport', 'dport']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Убираем служебные поля и оставляем только числовые признаки
drop_cols = ['pkSeqID', 'saddr', 'daddr', TARGET_COL]
X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
X = X.select_dtypes(include=['number']).fillna(0)
input_dim = X.shape[1]
print(f"Количество признаков: {input_dim}")

# Кодируем классы в числа 0..N-1
y = df[TARGET_COL].values
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
print(f"Классы ({num_classes}): {le.classes_}")

# Сначала train + temp, потом temp делим пополам на val/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X.values, y_encoded, test_size=0.3, random_state=RANDOM_STATE, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

Загрузка данных...
Количество признаков: 38
Классы (2): [0 1]


## Балансировка + масштабирование

Балансируем **только train** (чтобы не «подсматривать» в val/test), затем стандартизируем признаки и собираем `DataLoader`.

In [4]:
def apply_resampling(X, y, method, random_state):
    # Подбираем стратегию балансировки
    if method == 'undersample':
        sampler = RandomUnderSampler(random_state=random_state)
    elif method == 'oversample':
        sampler = RandomOverSampler(random_state=random_state)
    elif method == 'smote':
        sampler = SMOTE(random_state=random_state)
    else:
        return X, y

    X_res, y_res = sampler.fit_resample(X, y)
    print(f"Размер после {method}: {X_res.shape[0]} (было {X.shape[0]})")
    return X_res, y_res

X_train_res, y_train_res = apply_resampling(X_train, y_train, RESAMPLING_METHOD, RANDOM_STATE)

# Масштабирование обязательно после ресэмплинга
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

def to_tensor(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

X_train_t, y_train_t = to_tensor(X_train_scaled, y_train_res)
X_val_t, y_val_t = to_tensor(X_val_scaled, y_val)
X_test_t, y_test_t = to_tensor(X_test_scaled, y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

Размер после smote: 2697272 (было 1348970)


## Архитектура модели

Классический MLP: несколько полносвязных слоёв с `ReLU`, `BatchNorm` и `Dropout` для стабилизации обучения.

In [5]:
class DDoSClassifierMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DDoSClassifierMLP, self).__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        # Для инференса удобнее вернуть вероятности
        return torch.softmax(self.backbone(x), dim=1)

    def get_logits(self, x):
        # Для CrossEntropyLoss нужны "сырые" логиты
        return self.backbone(x)

## Обучение

Используем взвешенный `CrossEntropyLoss`, планировщик learning rate и `early stopping` по `val_loss`.

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")

model = DDoSClassifierMLP(input_dim=input_dim, num_classes=num_classes).to(device)

# Чем реже класс, тем больше его вклад в loss
class_counts = np.bincount(y_train_res)
class_weights = 1.0 / (class_counts + 1e-8)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

best_val_loss = float('inf')
trigger_times = 0

print("\nНачало обучения...")
epoch_pbar = tqdm(range(NUM_EPOCHS), desc="Epochs", leave=True)

for epoch in epoch_pbar:
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model.get_logits(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(logits.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # Валидация после каждой эпохи
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    class_correct = [0] * num_classes
    class_total = [0] * num_classes

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model.get_logits(inputs)
            loss = criterion(logits, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(logits.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            for i in range(len(predicted)):
                label = labels[i].item()
                pred = predicted[i].item()
                class_total[label] += 1
                if pred == label:
                    class_correct[label] += 1

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total
    scheduler.step(val_loss)

    per_class_acc = {}
    for c in range(num_classes):
        if class_total[c] > 0:
            per_class_acc[c] = 100 * class_correct[c] / class_total[c]
        else:
            per_class_acc[c] = float('nan')

    epoch_pbar.set_postfix(
        TrainLoss=f"{train_loss:.4f}",
        TrainAcc=f"{train_acc:.2f}%",
        ValLoss=f"{val_loss:.4f}",
        ValAcc=f"{val_acc:.2f}%"
    )

    # Раз в 5 эпох печатаем точность по каждому классу
    if (epoch + 1) % 5 == 0:
        msg = "Per-class val accuracy: " + ", ".join(
            [f"{le.classes_[c]}={per_class_acc[c]:.2f}%" for c in range(num_classes)]
        )
        epoch_pbar.write(msg)

    # Early stopping + сохранение лучшей модели
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        trigger_times = 0
        torch.save(model.state_dict(), 'best_ddos_model.pth')
    else:
        trigger_times += 1
        if trigger_times >= PATIENCE:
            epoch_pbar.write("Early stopping triggered.")
            break

epoch_pbar.close()

Устройство: cpu

Начало обучения...


Epochs:  50%|█████     | 5/10 [16:25<16:53, 202.68s/it, TrainAcc=100.00%, TrainLoss=0.0001, ValAcc=100.00%, ValLoss=0.0000]

Per-class val accuracy: 0=100.00%, 1=100.00%


Epochs: 100%|██████████| 10/10 [34:20<00:00, 206.02s/it, TrainAcc=100.00%, TrainLoss=0.0001, ValAcc=100.00%, ValLoss=0.0001]

Per-class val accuracy: 0=100.00%, 1=100.00%


## Тестирование и отчёт

Загружаем лучшие веса и считаем итоговую точность + `classification_report`.

In [7]:
print("\nОценка на тестовой выборке...")
model.load_state_dict(torch.load('best_ddos_model.pth', weights_only=True, map_location=device))
model.eval()

test_correct = 0
test_total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Test Accuracy: {100 * test_correct / test_total:.2f}%")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=[str(c) for c in le.classes_]))


Оценка на тестовой выборке...
Test Accuracy: 100.00%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        72
           1       1.00      1.00      1.00    288994

    accuracy                           1.00    289066
   macro avg       1.00      1.00      1.00    289066
weighted avg       1.00      1.00      1.00    289066

